# 🥇 Gold Layer: Real-Time Volatility Analytics
This notebook serves as the final consumption layer of our Medallion architecture. It utilizes **Hopping Windows** to calculate real-time price volatility and moving averages for **VOO, VYM, and VGT**, providing actionable insights from the streaming data.

### 📈 Analytics Logic
1. **Streaming Aggregation:** Reads from the Silver Delta table as a continuous stream.
2. **Watermarking:** Implements a 20-minute watermark to handle late-arriving data without losing state.
3. **Hopping Window:** Calculates metrics over a **10-minute window**, sliding every **2 minutes**.
4. **Volatility Metric:** Defined as the `max(High - Low)` within the window for each ticker.

In [2]:
import pyspark.sql.functions as f

# 1. Read the Silver table as a stream
silver_stream = (
    spark.readStream
    .format("delta")
    .table("silver_stock_prices")
)

# 2. Define the Hopping Window logic
# Window: 10 mins long, moves every 2 mins
gold_analysis = (
    silver_stream
    .withWatermark("event_time", "20 minutes")
    .groupBy(
        f.window(f.col("event_time"), "10 minutes", "2 minutes"),
        f.col("ticker")
    )
    .agg(
        f.max(f.col("High") - f.col("Low")).alias("max_volatility"),
        f.avg("Close").alias("avg_price"),
        f.count("ticker").alias("record_count")
    )
    .select(
        f.col("window.start").alias("window_start"),
        f.col("window.end").alias("window_end"),
        "ticker",
        "max_volatility",
        "avg_price",
        "record_count"
    )
)

# 3. Write to Gold Delta Table
query = (
    gold_analysis.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", "Files/checkpoints/gold_volatility")
    .toTable("gold_stock_volatility")
)

StatementMeta(, 5fbe29e6-4856-4b57-9f89-966bcd2080eb, 3, Finished, Available, Finished, False)

In [3]:
# Run this in a new cell to see the volatility data
display(spark.read.table("gold_stock_volatility").sort(f.col("window_start").desc()))

StatementMeta(, 5fbe29e6-4856-4b57-9f89-966bcd2080eb, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ca2153f6-2971-445e-8b94-60fda301ee35)